In [1]:
import os
from pathlib import Path
from collections import defaultdict
from PIL import Image
import numpy as np


def build_fast_structure(root_dir):
    root = Path(root_dir)
    data = defaultdict(lambda: defaultdict(lambda: {"frames": [], "num_frames": 0}))

    for png_path in root.rglob("*.png"):
        parts = png_path.parts
        class_folder = parts[1]
        
        filename = png_path.stem

        video_name, frame_str = filename.split("_x264_")
        frame_num = int(frame_str)

        data[class_folder][video_name]["frames"].append((frame_num, png_path))

    # Sort frames by frame number and store only paths
    for class_name in data:
        for video_name in data[class_name]:
            frames = data[class_name][video_name]["frames"]
            frames_sorted = [p for _, p in sorted(frames)]
            data[class_name][video_name]["frames"] = frames_sorted
            data[class_name][video_name]["num_frames"] = len(frames_sorted)

    return {cls: dict(videos) for cls, videos in data.items()}


def pick_16_frames(frame_list):
    if len(frame_list) == 0:
        return []

    if len(frame_list) <= 16:
        return frame_list + [frame_list[-1]] * (16 - len(frame_list))

    idx = np.linspace(0, len(frame_list) - 1, 16).astype(int)
    return [frame_list[i] for i in idx]


def tile_images(image_paths, output_path):
    tile_size = 64
    grid_size = 4

    final_img = Image.new("RGB", (tile_size * grid_size, tile_size * grid_size))
    for i, path in enumerate(image_paths):
        img = Image.open(path).convert("RGB")
        img = img.resize((tile_size, tile_size), Image.NEAREST)

        row = i // grid_size
        col = i % grid_size
        final_img.paste(img, (col * tile_size, row * tile_size))
    final_img.save(output_path)


def process_dataset(root_dir, output_root):
    structure = build_fast_structure(root_dir)

    for class_name, videos in structure.items():
        for video_name, info in videos.items():
            frame_list = info["frames"]
            frames_16 = pick_16_frames(frame_list)

            out_dir = Path(output_root) / class_name
            out_dir.mkdir(parents=True, exist_ok=True)
            output_png = out_dir / f"{video_name}.png"
            tile_images(frames_16, output_png)